<a href="https://colab.research.google.com/github/zainabkhalid663/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabkhalid663/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring (regression-style, continuous 0–1 opportunity score).
Not classification — "should refresh" isn't binary, it's a spectrum. A page at
position 4 with fading impressions and a page at position 18 with rising
impressions both deserve different scores, not the same yes/no bucket.
Score doubles as a ranking once sorted, so I can hand off a prioritized list.

In [ ]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Proxy, not observed outcome. No historical "we refreshed X and it gained Y
traffic" data exists yet, so I can't train on a real label. Instead I define
opportunity from signals that predict it: ranking position (striking
distance, ~5-20), impression volume, CTR vs expected CTR for that position,
and traffic trend (declining/flat vs growing). Combined into a rule-based
proxy score now — this becomes the training target once real
before/after refresh data exists.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Hit rate on the top-N: of the pages my score ranks highest, what % actually
show the proxy signal cluster (declining trend + striking-distance position)
on manual check. Target 10-50% hit rate, same bar I used for lane
validation in ML-02. Longer-term metric once refreshes start shipping:
correlation between score and actual traffic lift 30/60 days post-refresh.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
!git clone -q https://github.com/zainabkhalid663/Flyrank-ML-Internship.git
import pandas as pd
df = pd.read_csv("Flyrank-ML-Internship/data/raw/content_refresh_anonymized.csv")

# check hit rate: how much of inventory falls in the "opportunity" band my proxy defines
striking_distance = (df["avg_position"] > 5) & (df["avg_position"] <= 20)
has_demand = df["impressions_90d"] >= 100
declining = df["trend_direction"] == "down"

flagged = (striking_distance & has_demand & declining).sum()
print(f"{flagged:,} / {len(df):,} pages ({flagged/len(df):.1%}) flagged as high opportunity")

7,703 / 30,000 pages (25.7%) flagged as high opportunity


In [4]:
import pandas as pd
df = pd.read_csv("Flyrank-ML-Internship/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df[["content_id", "avg_position", "impressions_90d", "ctr", "trend_direction", "trend_pct", "days_since_last_update"]].head()

(30000, 44)


,content_id,avg_position,impressions_90d,ctr,trend_direction,trend_pct,days_since_last_update
0,content_304f48230142,10.6,3803,0.76,down,-41.4,20
1,content_a1fb4e703a9e,20.3,15320,0.05,down,-57.7,25
2,content_9aa793d4d895,36.5,12581,0.09,down,-60.9,20
3,content_331d6c4de07b,6.2,11751,0.49,stable,-13.8,22
4,content_d99b7a2d90ca,44.0,19140,0.13,down,-34.7,14


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
# fixed rule: striking distance + demand + declining
rule_flagged = df[(df["avg_position"].between(5, 20)) & (df["impressions_90d"] >= 100) & (df["trend_direction"] == "down")]

# but look at what it misses: high-impression pages outside the position band
missed = df[(df["avg_position"] > 20) & (df["impressions_90d"] >= 1000) & (df["trend_direction"] == "down")]

print(f"rule flags: {len(rule_flagged):,}")
print(f"missed (position >20 but high demand + declining): {len(missed):,}")

rule flags: 7,783
missed (position >20 but high demand + declining): 2,200


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.